# Setup and Dependencies


In [1]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install timm efficientnet-pytorch
!pip install kaggle pandas scikit-learn opencv-python tqdm seaborn
!pip install torchmetrics
!pip install pydicom
!pip install --upgrade kaggle

import sys
print("✓ Dependencies installed!")
print(f"Python version: {sys.version}")

import torch
print(f"PyTorch version: {torch.__version__}")
# Use torch.version.cuda to get the CUDA toolkit version
print(f"CUDA toolkit version: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Looking in indexes: https://download.pytorch.org/whl/cu118
✓ Dependencies installed!
Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
PyTorch version: 2.10.0+cu128
CUDA toolkit version: 12.8
GPU: NVIDIA L4


# Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Create directory for saving models
import os
os.makedirs('/content/drive/MyDrive/EyeShield/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/EyeShield/logs', exist_ok=True)

print("✓ Google Drive mounted!")
print("Save location: /content/drive/MyDrive/EyeShield/")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive mounted!
Save location: /content/drive/MyDrive/EyeShield/


# Setup Kaggle API

In [3]:
from google.colab import files
import json
from pathlib import Path

print("Upload your kaggle.json file...")
print("Instructions:")
print("1. Go to: https://www.kaggle.com/settings/account")
print("2. Click 'Create New API Token'")
print("3. Upload the downloaded kaggle.json file")

uploaded = files.upload()

if 'kaggle.json' in uploaded:
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(exist_ok=True)

    with open(kaggle_dir / 'kaggle.json', 'w') as f:
        f.write(uploaded['kaggle.json'].decode())

    os.chmod(kaggle_dir / 'kaggle.json', 0o600)
    print("✓ Kaggle API configured!")
else:
    print("⚠ kaggle.json not found. You can continue with sample data.")

Upload your kaggle.json file...
Instructions:
1. Go to: https://www.kaggle.com/settings/account
2. Click 'Create New API Token'
3. Upload the downloaded kaggle.json file


Saving kaggle.json to kaggle (1).json
⚠ kaggle.json not found. You can continue with sample data.


# Download Dataset from Kaggle

In [4]:
import kagglehub

# Unzip if needed
import zipfile
dataset_path = kagglehub.dataset_download("ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy")

for file in os.listdir(dataset_path):
    if file.endswith('.zip'):
        with zipfile.ZipFile(os.path.join(dataset_path, file), 'r') as zip_ref:
            zip_ref.extractall(dataset_path)

print(f"✓ Dataset downloaded to: {dataset_path}")
print(f"Files: {os.listdir(dataset_path)[:10]}")


Using Colab cache for faster access to the 'eyepacs-aptos-messidor-diabetic-retinopathy' dataset.
✓ Dataset downloaded to: /kaggle/input/eyepacs-aptos-messidor-diabetic-retinopathy
Files: ['dr_unified_v2', 'augmented_resized_V2']


# Copy Training Script

In [6]:
!wget -O /content/eyeshield_training_preprocessor.py \
  "https://raw.githubusercontent.com/dondondon22/EyeShield/refs/heads/main/eyeshield_training_preprocessor.py"

# OR if using GitHub Gist:
# !wget -O /content/eyeshield_sprint3_training.py "https://gist.githubusercontent.com/..."

print("✓ Training script downloaded!")

--2026-03-08 02:17:12--  https://raw.githubusercontent.com/dondondon22/EyeShield/refs/heads/main/eyeshield_training_preprocessor.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 30744 (30K) [text/plain]
Saving to: ‘/content/eyeshield_training_preprocessor.py’

/content/eyeshield_ 100%[===================>]  30.02K  --.-KB/s    in 0.002s  

2026-03-08 02:17:12 (14.5 MB/s) - ‘/content/eyeshield_training_preprocessor.py’ saved [30744/30744]

✓ Training script downloaded!


# Prepare Dataset CSV

In [7]:
import pandas as pd
import os
from pathlib import Path

# The dataset was downloaded to `dataset_path` variable in the previous step.
# Use this variable for the root directory of the dataset.
dataset_root = kagglehub.dataset_download("ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy") # Re-define for isolated execution

# Create directory for the CSV file if it doesn't exist
os.makedirs('/content/dataset', exist_ok=True)

images = []
labels = []

print(f"DEBUG: Dataset root: {dataset_root}")
print(f"DEBUG: Contents of dataset root: {os.listdir(dataset_root)}")

# Only use images from dr_unified_v2 subdirectory
dr_unified_v2_path = os.path.join(dataset_root, 'dr_unified_v2')

if os.path.isdir(dr_unified_v2_path):
    print(f"DEBUG: Processing only dr_unified_v2 subdirectory: {dr_unified_v2_path}")

    # Check if there's a nested dr_unified_v2 directory
    nested_dr_path = os.path.join(dr_unified_v2_path, 'dr_unified_v2')
    if os.path.isdir(nested_dr_path):
        print(f"DEBUG: Found nested dr_unified_v2, using: {nested_dr_path}")
        dr_unified_v2_path = nested_dr_path

    # Look for 'train', 'test', 'validation' directories
    for data_split in os.listdir(dr_unified_v2_path):
        data_split_path = os.path.join(dr_unified_v2_path, data_split)
        if os.path.isdir(data_split_path):
            print(f"DEBUG:   Processing data split: {data_split_path}")
            # Now look for the actual DR level directories (0, 1, 2, 3, 4)
            for dr_level in os.listdir(data_split_path):
                level_dir = os.path.join(data_split_path, dr_level)
                if os.path.isdir(level_dir):
                    try:
                        level_int = int(dr_level)
                        # print(f"DEBUG:     Processing DR level directory: {level_dir}")
                        found_images_in_level = False
                        for img_file in os.listdir(level_dir):
                            if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                                full_img_path = os.path.join(level_dir, img_file)
                                images.append(os.path.relpath(full_img_path, dataset_root))
                                labels.append(level_int)
                                found_images_in_level = True
                        if not found_images_in_level:
                            print(f"DEBUG:       No images found in {level_dir} or they don't match extensions.")
                    except ValueError:
                        print(f"DEBUG:       Skipping non-integer directory: {dr_level}")
                else:
                    print(f"DEBUG:       {level_dir} is not a directory. Skipping.")
        else:
            print(f"DEBUG:     {data_split_path} is not a directory. Skipping.")
else:
    print(f"ERROR: dr_unified_v2 subdirectory not found at {dr_unified_v2_path}")

df = pd.DataFrame({
    'image_path': images,
    'diagnosis': labels
})

df.to_csv('/content/dataset/labels.csv', index=False)

print(f"✓ Dataset CSV created!")
print(f"Total images: {len(df)}")
print(f"Class distribution:\n{df['diagnosis'].value_counts().sort_index()}")
print(f"\nSample:\n{df.head()}")

Using Colab cache for faster access to the 'eyepacs-aptos-messidor-diabetic-retinopathy' dataset.
DEBUG: Dataset root: /kaggle/input/eyepacs-aptos-messidor-diabetic-retinopathy
DEBUG: Contents of dataset root: ['dr_unified_v2', 'augmented_resized_V2']
DEBUG: Processing only dr_unified_v2 subdirectory: /kaggle/input/eyepacs-aptos-messidor-diabetic-retinopathy/dr_unified_v2
DEBUG: Found nested dr_unified_v2, using: /kaggle/input/eyepacs-aptos-messidor-diabetic-retinopathy/dr_unified_v2/dr_unified_v2
DEBUG:   Processing data split: /kaggle/input/eyepacs-aptos-messidor-diabetic-retinopathy/dr_unified_v2/dr_unified_v2/val
DEBUG:   Processing data split: /kaggle/input/eyepacs-aptos-messidor-diabetic-retinopathy/dr_unified_v2/dr_unified_v2/test
DEBUG:   Processing data split: /kaggle/input/eyepacs-aptos-messidor-diabetic-retinopathy/dr_unified_v2/dr_unified_v2/train
✓ Dataset CSV created!
Total images: 92501
Class distribution:
diagnosis
0    68953
1     4634
2    15151
3     1259
4     2504


# Modify Config and Run Training

In [8]:
# Read the training script
with open('/content/eyeshield_training_preprocessor.py', 'r') as f:
    training_code = f.read()

# Modify paths in the Config class (if needed)
modified_code = training_code.replace(
    "CHECKPOINT_DIR = './checkpoints'",
    "CHECKPOINT_DIR = '/content/drive/MyDrive/EyeShield/checkpoints'"
)

modified_code = modified_code.replace(
    "LOG_DIR = './logs'",
    "LOG_DIR = '/content/drive/MyDrive/EyeShield/logs'"
)

# Save modified script
with open('/content/eyeshield_training_preprocessor_modified.py', 'w') as f:
    f.write(modified_code)

print("✓ Configuration updated!")
print("Ready to start training...")

✓ Configuration updated!
Ready to start training...


# Backup

In [9]:

# Setup Auto-Backup on Colab Interruption
import shutil
import signal
import atexit
from datetime import datetime as dt

backup_executed = False

def backup_all_files():
    """Backup CSV and training artifacts to Google Drive"""
    global backup_executed
    if backup_executed:
        return

    backup_executed = True
    print("\n" + "="*80)
    print("EXECUTING BACKUP TO GOOGLE DRIVE")
    print("="*80)

    try:
        # Backup CSV to Drive
        csv_source = '/content/dataset/labels.csv'
        csv_dest = '/content/drive/MyDrive/EyeShield/labels_backup.csv'

        if os.path.exists(csv_source):
            shutil.copy2(csv_source, csv_dest)
            print(f"✓ Backed up CSV: {csv_dest}")
        else:
            print(f"⚠ CSV not found at {csv_source}")

        # Backup logs directory
        logs_source = '/content/logs'
        logs_dest = '/content/drive/MyDrive/EyeShield/logs_backup'

        if os.path.exists(logs_source):
            if os.path.exists(logs_dest):
                shutil.rmtree(logs_dest)
            shutil.copytree(logs_source, logs_dest)
            print(f"✓ Backed up logs: {logs_dest}")

        # Backup modified training script
        script_source = '/content/eyeshield_training_preprocessor_modified.py'
        script_dest = '/content/drive/MyDrive/EyeShield/training_script_backup.py'

        if os.path.exists(script_source):
            shutil.copy2(script_source, script_dest)
            print(f"✓ Backed up training script: {script_dest}")

        backup_time = dt.now().strftime('%Y-%m-%d %H:%M:%S')
        print(f"Backup timestamp: {backup_time}")
        print("="*80 + "\n")

    except Exception as e:
        print(f"❌ Backup error: {e}")
        import traceback
        traceback.print_exc()

def signal_handler(sig, frame):
    """Handle interruption signal"""
    print("\n⚠ Colab interruption detected! Running backup...")
    backup_all_files()
    raise KeyboardInterrupt

# Register backup to run on exit
atexit.register(backup_all_files)

# Register signal handlers for interruption
signal.signal(signal.SIGTERM, signal_handler)
signal.signal(signal.SIGINT, signal_handler)

print("✓ Auto-backup enabled on Colab stop/interruption")



✓ Auto-backup enabled on Colab stop/interruption


#  DEbug Training

In [10]:
# Debug: Verify all dependencies and paths exist

import os
import sys

print("=" * 80)
print("PRE-TRAINING CHECKS")
print("=" * 80)

# Check Python version
print(f"\n1. Python Version: {sys.version}")

# Check required packages
required_packages = ['torch', 'pandas', 'cv2', 'PIL', 'pydicom', 'sklearn']
print(f"\n2. Checking packages:")
for pkg in required_packages:
    try:
        __import__(pkg)
        print(f"   ✓ {pkg} available")
    except ImportError:
        print(f"   ✗ {pkg} NOT AVAILABLE")

# Check file paths
print(f"\n3. Checking file paths:")
files_to_check = [
    '/content/eyeshield_training_preprocessor_modified.py',
    '/content/dataset/labels.csv',
    '/content/drive/MyDrive/EyeShield/checkpoints',
    '/content/drive/MyDrive/EyeShield/logs'
]

for path in files_to_check:
    exists = os.path.exists(path)
    status = "✓" if exists else "✗"
    print(f"   {status} {path}")

# Check dataset CSV
try:
    import pandas as pd
    df = pd.read_csv('/content/dataset/labels.csv')
    print(f"\n4. Dataset CSV:")
    print(f"   ✓ Loaded {len(df)} records")
    print(f"   Columns: {list(df.columns)}")
    print(f"   Class distribution:\n{df['diagnosis'].value_counts().sort_index()}")
except Exception as e:
    print(f"\n4. Dataset CSV: ✗ Error - {e}")

print("\n" + "=" * 80)


PRE-TRAINING CHECKS

1. Python Version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]

2. Checking packages:
   ✓ torch available
   ✓ pandas available
   ✓ cv2 available
   ✓ PIL available
   ✓ pydicom available
   ✓ sklearn available

3. Checking file paths:
   ✓ /content/eyeshield_training_preprocessor_modified.py
   ✓ /content/dataset/labels.csv
   ✓ /content/drive/MyDrive/EyeShield/checkpoints
   ✓ /content/drive/MyDrive/EyeShield/logs

4. Dataset CSV:
   ✓ Loaded 92501 records
   Columns: ['image_path', 'diagnosis']
   Class distribution:
diagnosis
0    68953
1     4634
2    15151
3     1259
4     2504
Name: count, dtype: int64



In [ ]:

# Manual Backup (Run Anytime)
backup_all_files()
print("Manual backup completed. You can run this cell anytime to backup current progress.")


# TRAIN

In [ ]:
# Execute the full training pipeline

%cd /content

import sys
sys.path.insert(0, '/content')

# Import all training components
print("Loading training script...")
try:
    # Read and execute the modified training script
    with open('/content/eyeshield_training_preprocessor_modified.py', 'r') as f:
        training_script = f.read()

    # Create a clean namespace for execution
    exec_namespace = {'__name__': '__main__', '__file__': '/content/eyeshield_training_preprocessor_modified.py'}

    # Execute the script
    exec(training_script, exec_namespace)

    print("\n✓ Training completed!")

except FileNotFoundError as e:
    print(f"❌ File not found: {e}")
    print("   Make sure the 'Modify Config and Run Training' cell was executed first")
except Exception as e:
    print(f"❌ Error: {type(e).__name__}: {str(e)}")
    import traceback
    print("\nFull traceback:")
    traceback.print_exc()

finally:
    # Always backup on training completion or error
    print("\n⏳ Running final backup...")
    backup_all_files()
    print("✓ All files backed up to Google Drive")

/content
Loading training script...
Device: cuda
CUDA Available: True
GPU: NVIDIA L4

EyeShield: DR Classification Model Training (Sprint 3)
Using Your Image Preprocessor (No CLAHE)
Configuration:
  - Num Classes: 5
  - Target Preprocessing Size: (512, 512)
  - Input Size: (512, 512)
  - Batch Size: 32
  - Num Epochs: 100
  - Learning Rate: 0.001
  - EDL KL Weight: 0.1
  - Quality Check: True

Initializing image preprocessor...
✓ Image preprocessor initialized
  - Target size: (512, 512)
  - Quality assessment: Enabled

Loading dataset from CSV...
Using Colab cache for faster access to the 'eyepacs-aptos-messidor-diabetic-retinopathy' dataset.
✓ Loaded 92501 images from dataset
  - Class distribution:
diagnosis
0    68953
1     4634
2    15151
3     1259
4     2504
Name: count, dtype: int64

Data split:
  - Train: 64750 images
  - Val: 13875 images
  - Test: 13876 images

Initializing model...
Total Parameters: 11,091,511
Trainable Parameters: 11,091,511


Starting Training: EfficientN

Validation: 100%|██████████| 434/434 [01:59<00:00,  3.62it/s]



Epoch 1/100
Train Loss: 0.6083 | Train Acc: 0.7855
Val Loss: 1.1356 | Val Acc: 0.8251 | ECE: 0.4782
Best model saved: /content/drive/MyDrive/EyeShield/checkpoints/best_model.pt


Validation: 100%|██████████| 434/434 [01:59<00:00,  3.63it/s]



Epoch 2/100
Train Loss: 0.4973 | Train Acc: 0.8555
Val Loss: 1.0793 | Val Acc: 0.9707 | ECE: 0.4777
Best model saved: /content/drive/MyDrive/EyeShield/checkpoints/best_model.pt


...

Validation: 100%|██████████| 434/434 [01:59<00:00,  3.62it/s]



Epoch 16/100
Train Loss: 1.1022 | Train Acc: 0.9104
Val Loss: 1.0674 | Val Acc: 0.9759 | ECE: 0.5566


Validation: 100%|██████████| 434/434 [01:59<00:00,  3.62it/s]



Epoch 17/100
Train Loss: 1.1013 | Train Acc: 0.9120
Val Loss: 1.0633 | Val Acc: 0.9849 | ECE: 0.5674


Epoch 18/100:  30%|███       | 610/2024 [05:16<12:10,  1.94it/s, loss=1.12, nll=0.445, kl=6.72]

# RESUME TRAINING

In [ ]:
# Resume Training from Best Checkpoint
import os
import glob
import torch

checkpoint_dir = '/content/drive/MyDrive/EyeShield/checkpoints'

print("="*80)
print("RESUME TRAINING FROM CHECKPOINT")
print("="*80)

# List available checkpoints
checkpoint_files = glob.glob(os.path.join(checkpoint_dir, '*.pt'))
checkpoint_files.sort(key=os.path.getmtime)

print(f"\n📁 Checkpoint directory: {checkpoint_dir}")
print(f"Found {len(checkpoint_files)} checkpoint(s):\n")

best_model_path = os.path.join(checkpoint_dir, 'best_model.pt')
if os.path.exists(best_model_path):
    # Get modification time
    import datetime
    mtime = os.path.getmtime(best_model_path)
    mtime_str = datetime.datetime.fromtimestamp(mtime).strftime('%Y-%m-%d %H:%M:%S')
    size_mb = os.path.getsize(best_model_path) / (1024**2)
    print(f"✓ best_model.pt (BEST) - Modified: {mtime_str}, Size: {size_mb:.2f}MB")

for cp in checkpoint_files:
    if 'best_model' not in cp:
        mtime = os.path.getmtime(cp)
        mtime_str = datetime.datetime.fromtimestamp(mtime).strftime('%Y-%m-%d %H:%M:%S')
        size_mb = os.path.getsize(cp) / (1024**2)
        print(f"  • {os.path.basename(cp)} - Modified: {mtime_str}, Size: {size_mb:.2f}MB")

# Load best model checkpoint with weights_only=False for PyTorch 2.6+ compatibility
print(f"\n⏳ Loading best_model.pt...")
try:
    checkpoint = torch.load(best_model_path, map_location='cpu', weights_only=False)

    print(f"✓ Checkpoint loaded successfully!")
    print(f"  - Trained for {checkpoint.get('epoch', 'unknown')} epochs")
    print(f"  - Validation metrics: {checkpoint.get('val_metrics', {})}")

    # Store checkpoint for use in training cell
    RESUME_CHECKPOINT = checkpoint
    RESUME_CHECKPOINT_PATH = best_model_path
    RESUME_FROM_EPOCH = checkpoint.get('epoch', 0) + 1

    print(f"\n✓ Ready to resume from epoch {RESUME_FROM_EPOCH}")
    print("  Run the training cell below to continue from this checkpoint")

except Exception as e:
    print(f"❌ Error loading checkpoint: {e}")
    print("\n💡 Troubleshooting:")
    print("   • Make sure checkpoint file exists at:", best_model_path)
    print("   • If using PyTorch 2.6+, weights_only parameter is used")
    print("   • Checkpoint must contain model_state, optimizer_state, and epoch keys")
    RESUME_CHECKPOINT = None

# Resume TRAINING

In [ ]:
# RESUME TRAINING from Checkpoint
# Make sure to run the previous cell first to load the checkpoint

%cd /content

if 'RESUME_CHECKPOINT' in locals() and RESUME_CHECKPOINT is not None:
    print("="*80)
    print("RESUMING TRAINING FROM CHECKPOINT")
    print("="*80)

    import sys
    sys.path.insert(0, '/content')

    try:
        # Create namespace with resume checkpoint
        exec_namespace = {
            '__name__': '__main__',
            '__file__': '/content/eyeshield_training_preprocessor_modified.py',
            'RESUME_CHECKPOINT': RESUME_CHECKPOINT,
        }

        # Read the training script
        with open('/content/eyeshield_training_preprocessor_modified.py', 'r') as f:
            training_code = f.read()

        # Inject helper function that will be called from the training script
        # This function wraps trainer.train() to load checkpoint first
        helper_code = '''
def _resume_wrapper_train(trainer, resume_checkpoint=None):
    """Wrapper to load checkpoint before training"""
    if resume_checkpoint is not None:
        print("\\n" + "="*80)
        print("LOADING CHECKPOINT FOR RESUME")
        print("="*80)
        try:
            resume_ckpt = resume_checkpoint
            trainer.model.load_state_dict(resume_ckpt['model_state'])
            trainer.optimizer.load_state_dict(resume_ckpt['optimizer_state'])
            resume_epoch = resume_ckpt.get('epoch', 0)
            print(f"✓ Checkpoint loaded from epoch {resume_epoch}")
            print(f"✓ Model weights and optimizer state restored")
            print(f"✓ Resuming training from epoch {resume_epoch + 1}")
            print("="*80 + "\\n")
        except Exception as e:
            print(f"⚠ Could not load checkpoint: {e}")

    # Call original train method
    trainer.train()

'''

        # Add the helper function to the namespace before executing
        exec(helper_code, exec_namespace)

        # Modify the training script to use the wrapper
        # Replace trainer.train() with _resume_wrapper_train(trainer, RESUME_CHECKPOINT)
        modified_code = training_code.replace(
            'trainer.train()',
            '_resume_wrapper_train(trainer, RESUME_CHECKPOINT)'
        )

        # Execute the modified script
        exec(modified_code, exec_namespace)

        print("\n✓ Training resumed and completed!")

    except FileNotFoundError as e:
        print(f"❌ File not found: {e}")
        print("   Make sure the 'Modify Config and Run Training' cell was executed first")
    except Exception as e:
        print(f"❌ Error: {type(e).__name__}: {str(e)}")
        import traceback
        print("\nFull traceback:")
        traceback.print_exc()

    finally:
        # Always backup on completion
        print("\n⏳ Running final backup...")
        backup_all_files()
        print("✓ All files backed up to Google Drive")

else:
    print("❌ No checkpoint loaded!")
    print("   Please run the 'Resume Training from Best Checkpoint' cell first to load a checkpoint")



In [ ]:
# Checkpoint Inspector (Optional - View Details Before Resuming)
import torch
import os
import json

checkpoint_path = '/content/drive/MyDrive/EyeShield/checkpoints/best_model.pt'

if os.path.exists(checkpoint_path):
    print("="*80)
    print("CHECKPOINT DETAILS")
    print("="*80)

    # Load with weights_only=False for PyTorch 2.6+ compatibility
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

    print(f"\n📋 Checkpoint Information:")
    print(f"   Path: {checkpoint_path}")
    print(f"   File size: {os.path.getsize(checkpoint_path) / (1024**2):.2f} MB")

    if 'epoch' in checkpoint:
        print(f"   Last trained epoch: {checkpoint['epoch']}")

    if 'val_metrics' in checkpoint:
        metrics = checkpoint['val_metrics']
        print(f"\n📊 Validation Metrics at last save:")
        for key, value in metrics.items():
            if isinstance(value, (int, float)):
                print(f"      {key}: {value:.4f}")

    if 'model_state' in checkpoint:
        model_size = sum(p.numel() for p in checkpoint['model_state'].values())
        print(f"\n🧠 Model:")
        print(f"      Total parameters: {model_size:,}")

    print(f"\n✓ This checkpoint is ready to resume from!")
    print(f"  Next epoch will be: {checkpoint.get('epoch', 0) + 1}")

else:
    print(f"❌ Checkpoint not found at: {checkpoint_path}")
    print("   Train a model first or check the path")

# Monitor Training (Optional)

In [ ]:
# Use tensorboard to monitor training in real-time

# %load_ext tensorboard
# %tensorboard --logdir /content/logs

# Download Results

In [ ]:
from google.colab import files

# Download best model
files.download('/content/drive/MyDrive/EyeShield/checkpoints/best_model.pt')

# Download training history plot
files.download('/content/drive/MyDrive/EyeShield/logs/training_history.png')

# Download all checkpoints as zip
!cd /content/drive/MyDrive/EyeShield && zip -r checkpoints.zip checkpoints/
files.download('/content/drive/MyDrive/EyeShield/checkpoints.zip')

print("✓ Results downloaded!")

# Evaluation and Testing

In [ ]:
# Load Model Definition and Classes for Evaluation (Safe - No Training)
import sys
import torch
import re

sys.path.insert(0, '/content')

# Import the training module to get model class and config
print("Loading model class from training script...")
print("⚠ Extracting only class definitions (no training will execute)")

try:
    # Read the training script
    with open('/content/eyeshield_training_preprocessor_modified.py', 'r') as f:
        training_code = f.read()

    # Find and extract only the class definitions
    # Extract everything before "if __name__" or "trainer = Trainer" to avoid running training
    cutoff_markers = ['if __name__', 'trainer = Trainer', '# Train the model', 'trainer.train()']
    cutoff_index = len(training_code)

    for marker in cutoff_markers:
        idx = training_code.find(marker)
        if idx != -1 and idx < cutoff_index:
            cutoff_index = idx

    # Keep only the definitions part
    definitions_only = training_code[:cutoff_index]

    # Create a safe namespace for execution (only classes/functions, no side effects)
    model_namespace = {
        '__name__': '__main__',
        'torch': torch,
        'nn': torch.nn,
        'np': __import__('numpy'),
        'pd': __import__('pandas'),
        'cv2': __import__('cv2'),
        'pydicom': __import__('pydicom'),
        'Image': __import__('PIL').Image,
        'transforms': __import__('torchvision.transforms', fromlist=['transforms']),
        'models': __import__('torchvision.models', fromlist=['models']),
    }

    # Execute only the class definitions (not the training code)
    exec(definitions_only, model_namespace)

    # Extract the model class and config
    EfficientNetB3EDL = model_namespace.get('EfficientNetB3EDL')
    Config = model_namespace.get('Config')

    if EfficientNetB3EDL:
        print("✓ EfficientNetB3EDL class loaded successfully")
    else:
        print("⚠ EfficientNetB3EDL class not found")

    if Config:
        print("✓ Config class loaded successfully")
    else:
        print("⚠ Config class not found")

    # Set device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"✓ Using device: {device}")
    print("\n✓ Ready for evaluation - NO TRAINING EXECUTED")

except Exception as e:
    print(f"❌ Error loading model class: {e}")
    import traceback
    traceback.print_exc()



In [ ]:
# Create Test DataLoader for Evaluation
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import cv2
import numpy as np
from pathlib import Path

print("="*80)
print("CREATING TEST DATASET AND DATALOADER")
print("="*80)

try:
    # Read the dataset CSV
    csv_path = '/content/dataset/labels.csv'
    df = pd.read_csv(csv_path)

    print(f"\n✓ Loaded CSV with {len(df)} samples")
    print(f"  Class distribution:\n{df['diagnosis'].value_counts().sort_index()}")

    # Get dataset root from the CSV paths
    dataset_root = '/content/drive/MyDrive/EyeShield'

    # Check if we can infer the dataset root from stored paths
    # For now, we'll use the Kaggle dataset download path
    dataset_root_kaggle = '/root/.cache/kagglehub/datasets/ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy/versions/1'

    # Try to find the actual dataset root
    if not os.path.exists(dataset_root):
        # Try kagglehub cache
        import kagglehub
        try:
            dataset_root = kagglehub.dataset_download("ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy")
            print(f"✓ Found dataset at: {dataset_root}")
        except:
            print(f"⚠ Could not locate dataset root")

    # Simple Dataset class for evaluation
    class FundusImageDataset(Dataset):
        """Dataset class for fundus images"""
        def __init__(self, df, dataset_root, transform=None):
            self.df = df
            self.dataset_root = dataset_root
            self.transform = transform

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            img_path = os.path.join(self.dataset_root, row['image_path'])
            label = int(row['diagnosis'])

            # Load image
            if img_path.lower().endswith('.dcm'):
                import pydicom
                dicom = pydicom.dcmread(img_path)
                img = dicom.pixel_array
                if img.dtype != np.uint8:
                    img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
                if len(img.shape) == 2:
                    img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
            else:
                img = cv2.imread(img_path)
                if img is None:
                    # Return a placeholder if image not found
                    img = np.zeros((512, 512, 3), dtype=np.uint8)

            # Resize
            img = cv2.resize(img, (512, 512), interpolation=cv2.INTER_LANCZOS4)
            img = img.astype(np.float32) / 255.0

            # Apply transforms if provided
            if self.transform:
                img = self.transform(img)
            else:
                # Convert to tensor
                img = torch.from_numpy(img).permute(2, 0, 1)  # HWC -> CHW

            return img, torch.tensor(label, dtype=torch.long)

    # Create test dataset (using all data for evaluation)
    # In practice, you'd want a separate test split
    test_dataset = FundusImageDataset(df, dataset_root, transform=None)

    # Create test loader
    test_loader = DataLoader(
        test_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False
    )

    print(f"\n✓ Test DataLoader created!")
    print(f"  - Batch size: 32")
    print(f"  - Total batches: {len(test_loader)}")
    print(f"  - Device: {device}")
    print("\n✓ Ready for evaluation - Run the next cell to evaluate the model")

except Exception as e:
    print(f"❌ Error creating test loader: {e}")
    import traceback
    traceback.print_exc()



In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print("="*80)
print("MODEL EVALUATION AND TESTING")
print("="*80)

# Verify model class is available
if 'EfficientNetB3EDL' not in locals():
    print("⚠ Model class not loaded. Running setup cell...")
    print("Please run the 'Load Model Definition and Classes for Evaluation' cell first")
else:
    try:
        print("\n✓ Loading best model checkpoint...")
        checkpoint = torch.load('/content/drive/MyDrive/EyeShield/checkpoints/best_model.pt', weights_only=False)

        # Create and load model
        model = EfficientNetB3EDL(num_classes=5, pretrained=False)
        model.load_state_dict(checkpoint['model_state'])
        model.to(device)
        model.eval()

        print("✓ Model loaded successfully")
        print(f"  - Checkpoint epoch: {checkpoint.get('epoch', 'unknown')}")
        print(f"  - Model parameters: {sum(p.numel() for p in model.parameters()):,}")

        # NOTE: The test_loader should be defined from your data loading code
        # This is a placeholder - you need to create test_loader from your dataset

        print("\n⚠ NOTE: test_loader not yet defined")
        print("To evaluate the model, you need to:")
        print("  1. Load your test dataset")
        print("  2. Create a test_loader from it")
        print("  3. Then run the evaluation code below\n")

        # Placeholder evaluation code (will only work if test_loader is defined)
        if 'test_loader' in locals():
            print("Running evaluation on test set...")

            all_preds = []
            all_targets = []
            all_uncertainties = []

            with torch.no_grad():
                for images, targets in test_loader:
                    images = images.to(device)
                    targets = targets.to(device)

                    evidence = model(images)
                    output = model.predict(evidence)

                    all_preds.extend(output['pred'].cpu().numpy())
                    all_targets.extend(targets.cpu().numpy())
                    all_uncertainties.extend(output['uncertainty'].cpu().numpy())

            # Confusion Matrix
            cm = confusion_matrix(all_targets, all_preds)

            fig, ax = plt.subplots(figsize=(10, 8))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                        xticklabels=['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative'],
                        yticklabels=['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative'])
            ax.set_title('Confusion Matrix - DR Classification')
            ax.set_ylabel('True Label')
            ax.set_xlabel('Predicted Label')
            plt.tight_layout()
            plt.savefig('/content/drive/MyDrive/EyeShield/logs/confusion_matrix.png', dpi=300)
            plt.show()

            # Classification Report
            print("\nClassification Report:")
            print(classification_report(all_targets, all_preds,
                  target_names=['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']))

            # Uncertainty Distribution
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))

            axes[0].hist(all_uncertainties, bins=50, alpha=0.7)
            axes[0].set_title('Uncertainty Distribution')
            axes[0].set_xlabel('Uncertainty')
            axes[0].set_ylabel('Frequency')

            # Correct vs Incorrect predictions
            correct_unc = [u for u, p, t in zip(all_uncertainties, all_preds, all_targets) if p == t]
            incorrect_unc = [u for u, p, t in zip(all_uncertainties, all_preds, all_targets) if p != t]

            axes[1].hist(correct_unc, bins=30, alpha=0.7, label='Correct')
            axes[1].hist(incorrect_unc, bins=30, alpha=0.7, label='Incorrect')
            axes[1].set_title('Uncertainty: Correct vs Incorrect')
            axes[1].set_xlabel('Uncertainty')
            axes[1].set_ylabel('Frequency')
            axes[1].legend()

            plt.tight_layout()
            plt.savefig('/content/drive/MyDrive/EyeShield/logs/uncertainty_analysis.png', dpi=300)
            plt.show()

            print("\n✓ Evaluation complete!")
        else:
            print("To enable evaluation, create a test_loader from your test dataset")
            print("Example:")
            print("  from torch.utils.data import DataLoader")
            print("  test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)")

    except FileNotFoundError as e:
        print(f"❌ File not found: {e}")
        print("   Make sure checkpoints have been saved during training")
    except Exception as e:
        print(f"❌ Error: {type(e).__name__}: {str(e)}")
        import traceback
        traceback.print_exc()

print("\n" + "="*80)